In [ ]:
# train_compare_f1.py
"""
Train baseline models (ResNet50, EfficientNet-B4, ViT) using CrossEntropy,
use validation macro-F1 for early stopping (primary). Evaluate on test set,
then evaluate an existing hybrid checkpoint (no training).
No masking — uses standard ImageFolder layout: train/ val/ test/.
"""

import os
import copy
import time
import csv
from pathlib import Path

import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, cohen_kappa_score, classification_report, confusion_matrix
from sklearn.preprocessing import label_binarize
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD

# -----------------------------
# CONFIG - edit these
# -----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = r"D:\LIMUC"   # root containing train/, val/, test/ subfolders
OUTPUT_DIR = "checkpoints_f1_compare"
PLOT_DIR = "plots_f1_compare"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

HYBRID_CHECKPOINT = "best_reseff_fusion_final.pth"  # path to your hybrid checkpoint (optional)

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 4
NUM_EPOCHS = 100
LR = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 10   # early stopping waiting epochs on val macro-F1
PRETRAINED = True  # try pretrained models; fallback to False if download fails

# -----------------------------
# Safe timm creation helper
# -----------------------------
def safe_create_model(name, pretrained, num_classes=None):
    """
    Try to create timm model with pretrained flag. If it fails (HF or network),
    retry with pretrained=False. If num_classes provided and model supports
    num_classes argument, pass it (else create features-only wrapper below).
    """
    try:
        if num_classes is None:
            return timm.create_model(name, pretrained=pretrained)
        else:
            # Some timm models accept num_classes; for feature-only we will handle separately
            return timm.create_model(name, pretrained=pretrained, num_classes=num_classes)
    except Exception as e:
        print(f"Warning: timm.create_model({name}, pretrained={pretrained}) failed: {e}")
        print("Retrying with pretrained=False.")
        return timm.create_model(name, pretrained=False, num_classes=num_classes)

# -----------------------------
# Data loaders (no mask)
# -----------------------------
def build_dataloaders(data_root, img_size=IMG_SIZE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    train_dir = os.path.join(data_root, "train")
    val_dir = os.path.join(data_root, "val")
    test_dir = os.path.join(data_root, "test")
    if not (os.path.exists(train_dir) and os.path.exists(val_dir) and os.path.exists(test_dir)):
        raise RuntimeError(f"Expected train/ val/ test under {data_root}. Found: {os.listdir(data_root) if os.path.exists(data_root) else 'MISSING'}")

    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(0.1,0.1,0.1,0.05),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
    ])

    train_ds = ImageFolder(train_dir, transform=train_tf)
    val_ds = ImageFolder(val_dir, transform=eval_tf)
    test_ds = ImageFolder(test_dir, transform=eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size*2, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size*2, shuffle=False, num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds

# -----------------------------
# Model wrappers
# -----------------------------
class ResNetWrapper(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = safe_create_model('resnet50', pretrained=PRETRAINED, num_classes=num_classes)
    def forward(self, x): return self.model(x)

class EffNetWrapper(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = safe_create_model('efficientnet_b4', pretrained=PRETRAINED, num_classes=num_classes)
    def forward(self, x): return self.model(x)

class ViTWrapper(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = safe_create_model('vit_base_patch16_224', pretrained=PRETRAINED, num_classes=num_classes)
    def forward(self, x): return self.model(x)

def build_fusion_model(num_classes, eff_weight=0.75):
    """
    Thin fusion model similar to your hybrid: uses features_only backbones then projects.
    We create features_only backbones to avoid coupling classifiers.
    """
    class FusionModel(nn.Module):
        def __init__(self, num_classes, eff_weight):
            super().__init__()
            self.eff_weight = eff_weight
            self.res_weight = 1.0 - eff_weight
            # get feature backbones (pretrained possibly)
            try:
                self.eff = safe_create_model("efficientnet_b4", pretrained=PRETRAINED, num_classes=None)
                self.eff = timm.create_model("efficientnet_b4", pretrained=PRETRAINED, features_only=True)
            except Exception:
                self.eff = timm.create_model("efficientnet_b4", pretrained=False, features_only=True)
            try:
                self.res = timm.create_model("resnet50", pretrained=PRETRAINED, features_only=True)
            except Exception:
                self.res = timm.create_model("resnet50", pretrained=False, features_only=True)

            eff_dim = self.eff.feature_info[-1]['num_chs']
            res_dim = self.res.feature_info[-1]['num_chs']

            self.eff_proj = nn.Conv2d(eff_dim, 1024, kernel_size=1)
            self.res_proj = nn.Conv2d(res_dim, 1024, kernel_size=1)
            self.bn = nn.BatchNorm2d(1024)
            self.relu = nn.ReLU(inplace=False)
            self.pool = nn.AdaptiveAvgPool2d(1)
            self.classifier = nn.Linear(1024, num_classes)
            self.dropout = nn.Dropout(0.4)

        def forward(self, x):
            eff_feat = self.eff(x)[-1]
            res_feat = self.res(x)[-1]
            if eff_feat.shape[2:] != res_feat.shape[2:]:
                res_feat = nn.functional.interpolate(res_feat, size=eff_feat.shape[2:], mode='bilinear', align_corners=False)
            eff_f = self.eff_proj(eff_feat)
            res_f = self.res_proj(res_feat)
            fused = self.eff_weight * eff_f + self.res_weight * res_f
            fused = self.relu(self.bn(fused))
            pooled = self.pool(fused).flatten(1)
            pooled = self.dropout(pooled)
            return self.classifier(pooled)

    return FusionModel(num_classes, eff_weight)

# -----------------------------
# Training helper: optimize by val macro-F1
# -----------------------------
def train_one_model(model, name, train_loader, val_loader, num_epochs=NUM_EPOCHS, out_dir=OUTPUT_DIR):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))

    best_w = copy.deepcopy(model.state_dict())
    best_f1 = -1.0
    bad = 0
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_acc': []}

    for epoch in range(1, num_epochs+1):
        model.train()
        running_loss = 0.0
        n = 0
        pbar = tqdm(train_loader, desc=f"[{name}] Epoch {epoch}/{num_epochs}")
        for imgs, labels in pbar:
            imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * imgs.size(0)
            n += imgs.size(0)
            pbar.set_postfix({'loss': f"{running_loss/max(1,n):.4f}"})

        epoch_train_loss = running_loss / max(1,n)
        history['train_loss'].append(epoch_train_loss)

        # validation
        model.eval()
        y_true = []
        y_pred = []
        val_loss = 0.0
        v_n = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                probs = torch.softmax(outputs, dim=1)
                preds = probs.argmax(1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())
                val_loss += loss.item() * imgs.size(0)
                v_n += imgs.size(0)

        val_loss = val_loss / max(1, v_n)
        val_acc = float((np.array(y_true) == np.array(y_pred)).sum() / len(y_true))
        val_f1 = f1_score(y_true, y_pred, average='macro')
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        print(f"[{name}] Epoch {epoch} -> train_loss={epoch_train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")

        # scheduler step
        scheduler.step()

        # early stop on val_f1 (higher is better)
        if val_f1 > best_f1 + 1e-6:
            best_f1 = val_f1
            best_w = copy.deepcopy(model.state_dict())
            bad = 0
            torch.save(best_w, os.path.join(out_dir, f"{name}_best.pth"))
            print(f"Saved best {name} (val_f1={best_f1:.4f})")
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"[{name}] Early stopping at epoch {epoch} (no val_f1 improvement for {PATIENCE} epochs).")
                break

    # restore best weights
    model.load_state_dict(best_w)
    torch.save(model.state_dict(), os.path.join(out_dir, f"{name}_final.pth"))
    return model, history

# -----------------------------
# Evaluation (test metrics)
# -----------------------------
def evaluate_model(model, loader, class_names):
    model.to(DEVICE)
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.tolist())
            y_probs.extend(probs.tolist())

    y_true = np.array(y_true); y_pred = np.array(y_pred); y_probs = np.array(y_probs)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')

    report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    return {
        'accuracy': float(acc),
        'macro_f1': float(macro_f1),
        'precision': float(precision),
        'recall': float(recall),
        'qwk': float(qwk),
        'report': report,
        'confusion_matrix': cm,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_probs': y_probs
    }

# -----------------------------
# Main orchestration
# -----------------------------
def main():
    train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = build_dataloaders(DATA_ROOT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    CLASS_NAMES = train_ds.classes
    NUM_CLASSES = len(CLASS_NAMES)
    print("Detected classes:", CLASS_NAMES)

    # models to train
    models = {
        "ResNet50": ResNetWrapper(NUM_CLASSES),
        "EfficientNet-B4": EffNetWrapper(NUM_CLASSES),
        "ViT": ViTWrapper(NUM_CLASSES)
    }

    summary = []
    for name, model in models.items():
        print("\n" + "="*50)
        print(f"TRAINING {name}")
        print("="*50)
        trained_model, history = train_one_model(model, name, train_loader, val_loader, num_epochs=NUM_EPOCHS, out_dir=OUTPUT_DIR)
        print(f"Evaluating {name} on test set")
        stats = evaluate_model(trained_model, test_loader, CLASS_NAMES)

        # save confusion matrix plot
        cm = stats['confusion_matrix']
        plt.figure(figsize=(6,5))
        sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Blues')
        plt.title(f"{name} - Confusion Matrix")
        plt.savefig(os.path.join(PLOT_DIR, f"{name}_cm.png"))
        plt.close()

        summary.append({
            'model': name,
            'accuracy': stats['accuracy'],
            'macro_f1': stats['macro_f1'],
            'precision': stats['precision'],
            'recall': stats['recall'],
            'qwk': stats['qwk'],
            'params': sum(p.numel() for p in trained_model.parameters() if p.requires_grad)
        })
        # optionally save classification report
        with open(os.path.join(PLOT_DIR, f"{name}_report.txt"), "w") as f:
            f.write(stats['report'])

    # evaluate hybrid checkpoint if exists
    if os.path.exists(HYBRID_CHECKPOINT):
        print("\n" + "="*50)
        print("EVALUATING HYBRID (no training):", HYBRID_CHECKPOINT)
        # attempt to detect out dim
        sd = torch.load(HYBRID_CHECKPOINT, map_location='cpu')
        # try to infer out dim from weight named classifier.weight etc.
        out_dim = None
        sd_dict = sd if isinstance(sd, dict) else (sd.state_dict() if hasattr(sd, 'state_dict') else None)
        if isinstance(sd_dict, dict):
            for k,v in sd_dict.items():
                if k.endswith("classifier.weight") or k.endswith(".fc.weight") or k.endswith("head.weight"):
                    if isinstance(v, torch.Tensor):
                        out_dim = v.shape[0]
                        break
        if out_dim is None:
            print("Could not infer classifier output dim from hybrid checkpoint; attempting to load into flexible fusion model sized to NUM_CLASSES")
            hybrid_model = build_fusion_model(NUM_CLASSES)
            hybrid_model.load_state_dict(sd_dict, strict=False)
        else:
            # if out_dim==NUM_CLASSES assume standard classifier, else try to wrap
            if out_dim == NUM_CLASSES:
                try:
                    hybrid_model = build_fusion_model(NUM_CLASSES)
                    hybrid_model.load_state_dict(sd_dict, strict=False)
                except Exception:
                    hybrid_model = build_fusion_model(NUM_CLASSES)
                    hybrid_model.load_state_dict(sd_dict, strict=False)
            else:
                # load into fusion but keep strict=False
                hybrid_model = build_fusion_model(NUM_CLASSES)
                hybrid_model.load_state_dict(sd_dict, strict=False)

        stats_h = evaluate_model(hybrid_model, test_loader, CLASS_NAMES)
        summary.append({
            'model': 'Hybrid',
            'accuracy': stats_h['accuracy'],
            'macro_f1': stats_h['macro_f1'],
            'precision': stats_h['precision'],
            'recall': stats_h['recall'],
            'qwk': stats_h['qwk'],
            'params': sum(p.numel() for p in hybrid_model.parameters() if p.requires_grad)
        })
        with open(os.path.join(PLOT_DIR, "Hybrid_report.txt"), "w") as f:
            f.write(stats_h['report'])
    else:
        print("Hybrid checkpoint not found at:", HYBRID_CHECKPOINT)

    # Save final comparison sorted by macro_f1
    df = pd.DataFrame(summary).sort_values(by='macro_f1', ascending=False)
    print("\nFINAL COMPARISON (sorted by macro_f1):")
    print(df[['model','accuracy','macro_f1','qwk','params']].to_string(index=False))

    csv_out = os.path.join(OUTPUT_DIR, "models_f1_comparison.csv")
    df.to_csv(csv_out, index=False)
    print("Saved summary to:", csv_out)

if __name__ == "__main__":
    main()

Using MaskedImageFolder (masks detected).
Detected classes: ['0_normal', '1_ulcerative_colitis', '2_polyps', '3_esophagitis']


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4224\1841609963.py:239: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))



TRAINING ResNet50


[ResNet50] Epoch 1/100:   0%|          | 0/323 [00:00<?, ?it/s]

In [ ]:
# train_compare_f1.py
"""
Train baseline models (ResNet50, EfficientNet-B4, ViT) using CrossEntropy,
use validation macro-F1 for early stopping (primary). Evaluate on test set,
then evaluate an existing hybrid checkpoint (no training).
No masking — uses standard ImageFolder layout: train/ val/ test/.
"""

import os
import copy
import time
import csv
from pathlib import Path

import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, cohen_kappa_score, classification_report, confusion_matrix
from sklearn.preprocessing import label_binarize
from timm.data.constants import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD

# -----------------------------
# CONFIG - edit these
# -----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = r"D:\LIMUC"   # root containing train/, val/, test/ subfolders
OUTPUT_DIR = "checkpoints_f1_compare"
PLOT_DIR = "plots_f1_compare"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

HYBRID_CHECKPOINT = "best_reseff_fusion_final.pth"  # path to your hybrid checkpoint (optional)

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 4
NUM_EPOCHS = 100
LR = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 10   # early stopping waiting epochs on val macro-F1
PRETRAINED = True  # try pretrained models; fallback to False if download fails

# -----------------------------
# Safe timm creation helper
# -----------------------------
def safe_create_model(name, pretrained, num_classes=None):
    """
    Try to create timm model with pretrained flag. If it fails (HF or network),
    retry with pretrained=False. If num_classes provided and model supports
    num_classes argument, pass it (else create features-only wrapper below).
    """
    try:
        if num_classes is None:
            return timm.create_model(name, pretrained=pretrained)
        else:
            # Some timm models accept num_classes; for feature-only we will handle separately
            return timm.create_model(name, pretrained=pretrained, num_classes=num_classes)
    except Exception as e:
        print(f"Warning: timm.create_model({name}, pretrained={pretrained}) failed: {e}")
        print("Retrying with pretrained=False.")
        return timm.create_model(name, pretrained=False, num_classes=num_classes)

# -----------------------------
# Data loaders (no mask)
# -----------------------------
def build_dataloaders(data_root, img_size=IMG_SIZE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS):
    train_dir = os.path.join(data_root, "train")
    val_dir = os.path.join(data_root, "val")
    test_dir = os.path.join(data_root, "test")
    if not (os.path.exists(train_dir) and os.path.exists(val_dir) and os.path.exists(test_dir)):
        raise RuntimeError(f"Expected train/ val/ test under {data_root}. Found: {os.listdir(data_root) if os.path.exists(data_root) else 'MISSING'}")

    train_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(0.1,0.1,0.1,0.05),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD)
    ])

    train_ds = ImageFolder(train_dir, transform=train_tf)
    val_ds = ImageFolder(val_dir, transform=eval_tf)
    test_ds = ImageFolder(test_dir, transform=eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size*2, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size*2, shuffle=False, num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds

# -----------------------------
# Model wrappers
# -----------------------------
class ResNetWrapper(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = safe_create_model('resnet50', pretrained=PRETRAINED, num_classes=num_classes)
    def forward(self, x): return self.model(x)

class EffNetWrapper(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = safe_create_model('efficientnet_b4', pretrained=PRETRAINED, num_classes=num_classes)
    def forward(self, x): return self.model(x)

class ViTWrapper(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = safe_create_model('vit_base_patch16_224', pretrained=PRETRAINED, num_classes=num_classes)
    def forward(self, x): return self.model(x)

def build_fusion_model(num_classes, eff_weight=0.75):
    """
    Thin fusion model similar to your hybrid: uses features_only backbones then projects.
    We create features_only backbones to avoid coupling classifiers.
    """
    class FusionModel(nn.Module):
        def __init__(self, num_classes, eff_weight):
            super().__init__()
            self.eff_weight = eff_weight
            self.res_weight = 1.0 - eff_weight
            # get feature backbones (pretrained possibly)
            try:
                self.eff = safe_create_model("efficientnet_b4", pretrained=PRETRAINED, num_classes=None)
                self.eff = timm.create_model("efficientnet_b4", pretrained=PRETRAINED, features_only=True)
            except Exception:
                self.eff = timm.create_model("efficientnet_b4", pretrained=False, features_only=True)
            try:
                self.res = timm.create_model("resnet50", pretrained=PRETRAINED, features_only=True)
            except Exception:
                self.res = timm.create_model("resnet50", pretrained=False, features_only=True)

            eff_dim = self.eff.feature_info[-1]['num_chs']
            res_dim = self.res.feature_info[-1]['num_chs']

            self.eff_proj = nn.Conv2d(eff_dim, 1024, kernel_size=1)
            self.res_proj = nn.Conv2d(res_dim, 1024, kernel_size=1)
            self.bn = nn.BatchNorm2d(1024)
            self.relu = nn.ReLU(inplace=False)
            self.pool = nn.AdaptiveAvgPool2d(1)
            self.classifier = nn.Linear(1024, num_classes)
            self.dropout = nn.Dropout(0.4)

        def forward(self, x):
            eff_feat = self.eff(x)[-1]
            res_feat = self.res(x)[-1]
            if eff_feat.shape[2:] != res_feat.shape[2:]:
                res_feat = nn.functional.interpolate(res_feat, size=eff_feat.shape[2:], mode='bilinear', align_corners=False)
            eff_f = self.eff_proj(eff_feat)
            res_f = self.res_proj(res_feat)
            fused = self.eff_weight * eff_f + self.res_weight * res_f
            fused = self.relu(self.bn(fused))
            pooled = self.pool(fused).flatten(1)
            pooled = self.dropout(pooled)
            return self.classifier(pooled)

    return FusionModel(num_classes, eff_weight)

# -----------------------------
# Training helper: optimize by val macro-F1
# -----------------------------
def train_one_model(model, name, train_loader, val_loader, num_epochs=NUM_EPOCHS, out_dir=OUTPUT_DIR):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=='cuda'))

    best_w = copy.deepcopy(model.state_dict())
    best_f1 = -1.0
    bad = 0
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_acc': []}

    for epoch in range(1, num_epochs+1):
        model.train()
        running_loss = 0.0
        n = 0
        pbar = tqdm(train_loader, desc=f"[{name}] Epoch {epoch}/{num_epochs}")
        for imgs, labels in pbar:
            imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=='cuda')):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * imgs.size(0)
            n += imgs.size(0)
            pbar.set_postfix({'loss': f"{running_loss/max(1,n):.4f}"})

        epoch_train_loss = running_loss / max(1,n)
        history['train_loss'].append(epoch_train_loss)

        # validation
        model.eval()
        y_true = []
        y_pred = []
        val_loss = 0.0
        v_n = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(DEVICE); labels = labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                probs = torch.softmax(outputs, dim=1)
                preds = probs.argmax(1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())
                val_loss += loss.item() * imgs.size(0)
                v_n += imgs.size(0)

        val_loss = val_loss / max(1, v_n)
        val_acc = float((np.array(y_true) == np.array(y_pred)).sum() / len(y_true))
        val_f1 = f1_score(y_true, y_pred, average='macro')
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        print(f"[{name}] Epoch {epoch} -> train_loss={epoch_train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")

        # scheduler step
        scheduler.step()

        # early stop on val_f1 (higher is better)
        if val_f1 > best_f1 + 1e-6:
            best_f1 = val_f1
            best_w = copy.deepcopy(model.state_dict())
            bad = 0
            torch.save(best_w, os.path.join(out_dir, f"{name}_best.pth"))
            print(f"Saved best {name} (val_f1={best_f1:.4f})")
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"[{name}] Early stopping at epoch {epoch} (no val_f1 improvement for {PATIENCE} epochs).")
                break

    # restore best weights
    model.load_state_dict(best_w)
    torch.save(model.state_dict(), os.path.join(out_dir, f"{name}_final.pth"))
    return model, history

# -----------------------------
# Evaluation (test metrics)
# -----------------------------
def evaluate_model(model, loader, class_names):
    model.to(DEVICE)
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.tolist())
            y_probs.extend(probs.tolist())

    y_true = np.array(y_true); y_pred = np.array(y_pred); y_probs = np.array(y_probs)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')

    report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    return {
        'accuracy': float(acc),
        'macro_f1': float(macro_f1),
        'precision': float(precision),
        'recall': float(recall),
        'qwk': float(qwk),
        'report': report,
        'confusion_matrix': cm,
        'y_true': y_true,
        'y_pred': y_pred,
        'y_probs': y_probs
    }

# -----------------------------
# Main orchestration
# -----------------------------
def main():
    train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = build_dataloaders(DATA_ROOT, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    CLASS_NAMES = train_ds.classes
    NUM_CLASSES = len(CLASS_NAMES)
    print("Detected classes:", CLASS_NAMES)

    # models to train
    models = {
        "ResNet50": ResNetWrapper(NUM_CLASSES),
        "EfficientNet-B4": EffNetWrapper(NUM_CLASSES),
        "ViT": ViTWrapper(NUM_CLASSES)
    }

    summary = []
    for name, model in models.items():
        print("\n" + "="*50)
        print(f"TRAINING {name}")
        print("="*50)
        trained_model, history = train_one_model(model, name, train_loader, val_loader, num_epochs=NUM_EPOCHS, out_dir=OUTPUT_DIR)
        print(f"Evaluating {name} on test set")
        stats = evaluate_model(trained_model, test_loader, CLASS_NAMES)

        # save confusion matrix plot
        cm = stats['confusion_matrix']
        plt.figure(figsize=(6,5))
        sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Blues')
        plt.title(f"{name} - Confusion Matrix")
        plt.savefig(os.path.join(PLOT_DIR, f"{name}_cm.png"))
        plt.close()

        summary.append({
            'model': name,
            'accuracy': stats['accuracy'],
            'macro_f1': stats['macro_f1'],
            'precision': stats['precision'],
            'recall': stats['recall'],
            'qwk': stats['qwk'],
            'params': sum(p.numel() for p in trained_model.parameters() if p.requires_grad)
        })
        # optionally save classification report
        with open(os.path.join(PLOT_DIR, f"{name}_report.txt"), "w") as f:
            f.write(stats['report'])

    # evaluate hybrid checkpoint if exists
    if os.path.exists(HYBRID_CHECKPOINT):
        print("\n" + "="*50)
        print("EVALUATING HYBRID (no training):", HYBRID_CHECKPOINT)
        # attempt to detect out dim
        sd = torch.load(HYBRID_CHECKPOINT, map_location='cpu')
        # try to infer out dim from weight named classifier.weight etc.
        out_dim = None
        sd_dict = sd if isinstance(sd, dict) else (sd.state_dict() if hasattr(sd, 'state_dict') else None)
        if isinstance(sd_dict, dict):
            for k,v in sd_dict.items():
                if k.endswith("classifier.weight") or k.endswith(".fc.weight") or k.endswith("head.weight"):
                    if isinstance(v, torch.Tensor):
                        out_dim = v.shape[0]
                        break
        if out_dim is None:
            print("Could not infer classifier output dim from hybrid checkpoint; attempting to load into flexible fusion model sized to NUM_CLASSES")
            hybrid_model = build_fusion_model(NUM_CLASSES)
            hybrid_model.load_state_dict(sd_dict, strict=False)
        else:
            # if out_dim==NUM_CLASSES assume standard classifier, else try to wrap
            if out_dim == NUM_CLASSES:
                try:
                    hybrid_model = build_fusion_model(NUM_CLASSES)
                    hybrid_model.load_state_dict(sd_dict, strict=False)
                except Exception:
                    hybrid_model = build_fusion_model(NUM_CLASSES)
                    hybrid_model.load_state_dict(sd_dict, strict=False)
            else:
                # load into fusion but keep strict=False
                hybrid_model = build_fusion_model(NUM_CLASSES)
                hybrid_model.load_state_dict(sd_dict, strict=False)

        stats_h = evaluate_model(hybrid_model, test_loader, CLASS_NAMES)
        summary.append({
            'model': 'Hybrid',
            'accuracy': stats_h['accuracy'],
            'macro_f1': stats_h['macro_f1'],
            'precision': stats_h['precision'],
            'recall': stats_h['recall'],
            'qwk': stats_h['qwk'],
            'params': sum(p.numel() for p in hybrid_model.parameters() if p.requires_grad)
        })
        with open(os.path.join(PLOT_DIR, "Hybrid_report.txt"), "w") as f:
            f.write(stats_h['report'])
    else:
        print("Hybrid checkpoint not found at:", HYBRID_CHECKPOINT)

    # Save final comparison sorted by macro_f1
    df = pd.DataFrame(summary).sort_values(by='macro_f1', ascending=False)
    print("\nFINAL COMPARISON (sorted by macro_f1):")
    print(df[['model','accuracy','macro_f1','qwk','params']].to_string(index=False))

    csv_out = os.path.join(OUTPUT_DIR, "models_f1_comparison.csv")
    df.to_csv(csv_out, index=False)
    print("Saved summary to:", csv_out)

if __name__ == "__main__":
    main()

Classes: ['Mayo 0', 'Mayo 1', 'Mayo 2', 'Mayo 3']

Evaluating: ResNet50
ResNet50 → Acc: 0.7195 | F1: 0.6560 | QWK: 0.8090

Evaluating: EfficientNet-B4
EfficientNet-B4 → Acc: 0.7331 | F1: 0.6303 | QWK: 0.8033

Evaluating: ViT
ViT → Acc: 0.6133 | F1: 0.4346 | QWK: 0.4051

Evaluating: Hybrid
Hybrid → Acc: 0.7562 | F1: 0.6642 | QWK: 0.8272

FINAL COMPARISON (sorted by F1):
          model  accuracy  macro_f1      qwk
         Hybrid  0.756228  0.664164 0.827174
       ResNet50  0.719454  0.656025 0.808978
EfficientNet-B4  0.733096  0.630284 0.803307
            ViT  0.613286  0.434646 0.405125
